[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S10/S10_01_red_desde_cero.ipynb)

# S10 · Práctica 1 — Una red neuronal desde cero, solo con NumPy

**Tiempo estimado:** 1 h 15 min · **Sprint 10:** Redes neuronales bien entrenadas (UD 4.1)

**Contexto TurisData.** TurisData Canarias quiere saber si una red neuronal "vale la pena"
frente a los modelos clásicos que ya conoces. Antes de usar Keras, vamos a abrir la caja:
construirás una red diminuta con tus propias manos para entender **qué hace** por dentro
(así, cuando algo falle en Keras, sabrás por dónde mirar).

**Al terminar sabrás:**
1. Qué es una neurona artificial y para qué sirve la función de activación.
2. Cómo una red calcula una predicción (propagación hacia delante).
3. Cómo aprende: error, gradiente y propagación hacia atrás (*backpropagation*), sin fórmulas de memoria.
4. Entrenar una red pequeña sobre un problema que un modelo lineal no puede resolver.

In [ ]:
# Preparación (ejecuta esta celda siempre la primera)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_circles
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

SEMILLA = 42
np.random.seed(SEMILLA)
plt.rcParams["figure.figsize"] = (6, 4)

## 1. La neurona: una suma ponderada y un "interruptor suave"

**Idea.** Una neurona artificial hace tres cosas:
1. Multiplica cada entrada por un **peso** (cuánto le importa esa entrada).
2. Suma todo y añade un **sesgo** (*bias*, un empujón inicial).
3. Pasa el resultado por una **función de activación** (un "interruptor suave" que decide cuánta señal deja pasar).

Ejemplo de juguete: una neurona que estima la "probabilidad de cancelar" a partir de
la antelación (en cientos de días) y de si la tarifa es flexible (1) o no (0).

In [ ]:
def sigmoide(z):
    """Convierte cualquier número en un valor entre 0 y 1 (sirve como probabilidad)."""
    return 1 / (1 + np.exp(-z))

x = np.array([1.5, 1.0])        # antelación 150 días, tarifa flexible
pesos = np.array([0.8, 1.2])    # inventados: la tarifa flexible pesa más
sesgo = -1.0

z = np.dot(x, pesos) + sesgo    # suma ponderada
print("z =", z, "→ salida de la neurona =", round(float(sigmoide(z)), 3))

**Ejercicio 1.** Escribe la función `neurona(x, pesos, sesgo)` que devuelva la salida
de la neurona (suma ponderada + sesgo, pasada por `sigmoide`).

In [ ]:
# TODO: Usa np.dot(x, pesos) + sesgo y aplica sigmoide al resultado
...

salida_ejemplo = neurona(np.array([1.5, 1.0]), np.array([0.8, 1.2]), -1.0)
assert abs(salida_ejemplo - 0.8022) < 1e-3, "Revisa neurona(): debe ser sigmoide(x·pesos + sesgo). Esperado ≈ 0.802"
assert neurona(np.array([0.0, 0.0]), np.array([5.0, 5.0]), 0.0) == 0.5, "Con entradas 0 y sesgo 0 la sigmoide da 0.5"

## 2. Funciones de activación

**Idea.** Sin activación, apilar capas no sirve de nada: una suma de sumas ponderadas sigue
siendo una suma ponderada (un modelo lineal). La activación introduce **curvas**, y con curvas
la red puede dibujar fronteras complicadas.

- **Sigmoide:** salida entre 0 y 1. Típica en la capa final de una clasificación binaria.
- **ReLU:** deja pasar lo positivo y anula lo negativo. Es la habitual en capas ocultas.
- **Tanh:** como la sigmoide pero entre −1 y 1.

In [ ]:
def tanh(z):
    return np.tanh(z)

z = np.linspace(-6, 6, 200)
plt.plot(z, sigmoide(z), label="sigmoide")
plt.plot(z, tanh(z), label="tanh")
plt.plot(z, np.maximum(0, z), label="ReLU")
plt.ylim(-1.5, 3); plt.axhline(0, color="gray", lw=0.5); plt.axvline(0, color="gray", lw=0.5)
plt.xlabel("valor de entrada z"); plt.ylabel("salida de la activación")
plt.title("Tres funciones de activación"); plt.legend(); plt.show()

**Ejercicio 2.** Define `relu(z)` (devuelve `z` si es positivo y 0 si no) y `relu_derivada(z)`
(vale 1 donde `z > 0` y 0 en el resto). La derivada nos dirá más adelante "por dónde
fluye el aprendizaje": una neurona ReLU apagada (z ≤ 0) no aprende en esa muestra.

In [ ]:
# TODO: np.maximum(0, z) para relu; (z > 0) convertido a float para la derivada
...

_prueba = np.array([-2.0, 0.0, 3.0])
assert np.array_equal(relu(_prueba), [0.0, 0.0, 3.0]), "relu debe anular los negativos y dejar pasar los positivos"
assert np.array_equal(relu_derivada(_prueba), [0.0, 0.0, 1.0]), "relu_derivada vale 1 solo si z > 0"

## 3. Un problema que un modelo lineal no resuelve

Generamos dos "anillos" de puntos (uno dentro del otro). Piensa en ellos como clientes de
dos tipos con un comportamiento que no se separa con una línea recta. Una regresión logística
(modelo lineal) se quedará en el azar.

In [ ]:
X, y = make_circles(n_samples=600, noise=0.10, factor=0.45, random_state=SEMILLA)
X_ent, X_val, y_ent, y_val = train_test_split(X, y, test_size=0.3, random_state=SEMILLA, stratify=y)

lineal = LogisticRegression().fit(X_ent, y_ent)
print("Precisión de un modelo lineal en validación:", round(lineal.score(X_val, y_val), 3))

plt.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", s=12)
plt.title("Dos grupos que ninguna recta separa"); plt.xlabel("variable 1"); plt.ylabel("variable 2"); plt.show()

## 4. Propagación hacia delante: de las entradas a la predicción

Nuestra red tendrá: **2 entradas → capa oculta (ReLU) → 1 salida (sigmoide)**.
Una capa es simplemente **muchas neuronas a la vez**, y se calcula con una multiplicación de matrices:

`Z1 = X · W1 + b1` → `A1 = relu(Z1)` → `Z2 = A1 · W2 + b2` → `A2 = sigmoide(Z2)`

`A2` es la probabilidad predicha de pertenecer al grupo 1.

**Propagación hacia atrás.** Para aprender medimos el error (pérdida) y preguntamos:
*"si muevo un poquito este peso, ¿el error sube o baja, y cuánto?"*. Esa respuesta es el **gradiente**.
La regla de la cadena permite calcularlo capa por capa **desde la salida hacia la entrada**
(de ahí "hacia atrás"). Después movemos cada peso un pasito en dirección contraria al gradiente:
es el **descenso de gradiente**.

In [ ]:
def entropia_cruzada(y_real, prob, eps=1e-9):
    """Pérdida media de una clasificación binaria (menor = mejor)."""
    prob = np.clip(prob, eps, 1 - eps)
    return -np.mean(y_real * np.log(prob) + (1 - y_real) * np.log(1 - prob))

print("Si predices siempre 0.5, la pérdida es", round(entropia_cruzada(np.array([0, 1, 1, 0]), np.full(4, 0.5)), 3), "(= ln 2)")
print("Si aciertas con seguridad (0.99 / 0.01):", round(entropia_cruzada(np.array([1, 0]), np.array([0.99, 0.01])), 3))
print("Si fallas con seguridad (0.01 / 0.99):", round(entropia_cruzada(np.array([1, 0]), np.array([0.01, 0.99])), 3))

**Ejercicio 3.** Escribe `paso_gradiente(parametros, gradientes, tasa)`: recibe dos diccionarios con
las mismas claves (`"W1"`, `"b1"`, `"W2"`, `"b2"`) y devuelve un diccionario **nuevo** con cada
parámetro actualizado: `nuevo = viejo - tasa * gradiente`. `tasa` es la **tasa de aprendizaje**
(el tamaño del pasito).

In [ ]:
# TODO: Recorre las claves y resta tasa * gradiente a cada parámetro
...

_p = {"W1": np.array([1.0, 2.0]), "b1": np.array([0.5])}
_g = {"W1": np.array([0.5, -1.0]), "b1": np.array([1.0])}
_nuevo = paso_gradiente(_p, _g, 0.1)
assert np.allclose(_nuevo["W1"], [0.95, 2.1]) and np.allclose(_nuevo["b1"], [0.4]), "Cada parámetro debe restar tasa * gradiente"
assert _p["W1"][0] == 1.0, "No modifiques el diccionario original: devuelve uno nuevo"

## 5. La red completa

Aquí tienes la clase completa. Lee los comentarios: `adelante` es la propagación hacia delante,
`atras` calcula los gradientes y `entrenar` repite *adelante → atrás → actualizar* muchas veces
(cada vuelta completa a los datos es una **época**).

In [ ]:
class RedNeuronal:
    """Red de una capa oculta (ReLU) y salida sigmoide para clasificación binaria."""

    def __init__(self, n_entradas, n_ocultas, semilla=SEMILLA):
        rng = np.random.default_rng(semilla)
        self.p = {
            "W1": rng.normal(0, np.sqrt(2 / n_entradas), (n_entradas, n_ocultas)),
            "b1": np.zeros(n_ocultas),
            "W2": rng.normal(0, np.sqrt(1 / n_ocultas), (n_ocultas, 1)),
            "b2": np.zeros(1),
        }
        self.historial = []

    def adelante(self, X):
        p = self.p
        self.Z1 = X @ p["W1"] + p["b1"]
        self.A1 = relu(self.Z1)
        self.Z2 = self.A1 @ p["W2"] + p["b2"]
        self.A2 = sigmoide(self.Z2)
        return self.A2.ravel()

    def atras(self, X, y):
        m = len(X)
        dZ2 = self.A2 - y.reshape(-1, 1)                 # error de la salida
        g = {"W2": self.A1.T @ dZ2 / m, "b2": dZ2.mean(axis=0)}
        dA1 = dZ2 @ self.p["W2"].T                        # el error viaja hacia atrás...
        dZ1 = dA1 * relu_derivada(self.Z1)                # ...atravesando la ReLU
        g["W1"] = X.T @ dZ1 / m
        g["b1"] = dZ1.mean(axis=0)
        return g

    def entrenar(self, X, y, epocas=1500, tasa=0.3, X_val=None, y_val=None):
        for e in range(epocas):
            prob = self.adelante(X)
            g = self.atras(X, y)
            self.p = paso_gradiente(self.p, g, tasa)
            if e % 50 == 0 or e == epocas - 1:
                perdida_val = entropia_cruzada(y_val, self.adelante(X_val)) if X_val is not None else np.nan
                self.historial.append((e, entropia_cruzada(y, prob), perdida_val))
        return self

    def predecir(self, X):
        return (self.adelante(X) >= 0.5).astype(int)

**Comprobación de gradientes.** Una forma muy usada de asegurarse de que `atras` está bien es compararlo
con una estimación numérica: mover un peso una milésima y ver cuánto cambia la pérdida.
Si coinciden, la propagación hacia atrás es correcta.

In [ ]:
red_test = RedNeuronal(2, 4)
red_test.adelante(X_ent); g = red_test.atras(X_ent, y_ent)

def perdida_con_peso(red, clave, idx, valor):
    original = red.p[clave][idx]
    red.p[clave][idx] = valor
    perdida = entropia_cruzada(y_ent, red.adelante(X_ent))
    red.p[clave][idx] = original
    return perdida

idx, h = (0, 1), 1e-5
w = red_test.p["W1"][idx]
numerico = (perdida_con_peso(red_test, "W1", idx, w + h) - perdida_con_peso(red_test, "W1", idx, w - h)) / (2 * h)
print("Gradiente por backprop :", round(float(g["W1"][idx]), 6))
print("Gradiente numérico     :", round(float(numerico), 6))
assert abs(g["W1"][idx] - numerico) < 1e-4, "El gradiente de la red no coincide con el numérico"

## 6. Entrenamos y comparamos tamaños de capa oculta

Probaremos 1, 2 y 8 neuronas ocultas. Pregunta para ti antes de ejecutar: ¿cuántas neuronas crees
que hacen falta para separar dos anillos?

In [ ]:
resultados = {}
for n_ocultas in (1, 2, 8):
    red = RedNeuronal(2, n_ocultas).entrenar(X_ent, y_ent, epocas=1500, tasa=0.3, X_val=X_val, y_val=y_val)
    resultados[n_ocultas] = red
    print(f"{n_ocultas} neuronas ocultas → precisión validación: {(red.predecir(X_val) == y_val).mean():.3f}")

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(14, 4))
xx, yy = np.meshgrid(np.linspace(-1.6, 1.6, 200), np.linspace(-1.6, 1.6, 200))
rejilla = np.c_[xx.ravel(), yy.ravel()]
for eje, (n, red) in zip(ejes, resultados.items()):
    zona = red.adelante(rejilla).reshape(xx.shape)
    eje.contourf(xx, yy, zona, levels=20, cmap="coolwarm", alpha=0.6)
    eje.scatter(X_val[:, 0], X_val[:, 1], c=y_val, cmap="coolwarm", edgecolor="k", s=15)
    eje.set_title(f"{n} neurona(s) oculta(s)"); eje.set_xlabel("variable 1"); eje.set_ylabel("variable 2")
plt.tight_layout(); plt.show()

In [ ]:
mejor = resultados[8]
epocas_h, perd_ent, perd_val = zip(*mejor.historial)
plt.plot(epocas_h, perd_ent, label="entrenamiento")
plt.plot(epocas_h, perd_val, label="validación")
plt.xlabel("época"); plt.ylabel("pérdida (entropía cruzada)")
plt.title("La pérdida baja: la red está aprendiendo"); plt.legend(); plt.show()

**Ejercicio 4.** Calcula en la variable `precision_val` la precisión de la red de 8 neuronas
(`mejor`) sobre `X_val`/`y_val` usando su método `predecir`. Y en `mejora_sobre_lineal`
guarda cuántos puntos porcentuales (en fracción, p. ej. 0.4) supera al modelo `lineal`.

In [ ]:
# TODO: precisión = media de aciertos (predecir(X_val) == y_val); compara con lineal.score
...

assert precision_val > 0.9, "La red de 8 neuronas debería superar el 90 %. ¿Has usado predecir(X_val) y comparado con y_val?"
assert mejora_sobre_lineal > 0.25, "La red debería superar claramente al modelo lineal (≈ 50 %)"

## 7. Mini-reto integrador

Entrena una red con **4 neuronas ocultas** y una **tasa de aprendizaje de 0.01** (muy pequeña) durante 1500 épocas,
guarda la red en `red_lenta` y su precisión de validación en `precision_lenta`.
Después contesta en la celda de texto: ¿por qué le cuesta tanto más? ¿Qué pasaría con una tasa de 50?

In [ ]:
# TODO: RedNeuronal(2, 4).entrenar(..., tasa=0.01) y mide la precisión igual que antes
...

assert red_lenta.historial[-1][0] == 1499, "Entrena 1500 épocas (la última registrada es la 1499)"
assert precision_lenta < precision_val + 1e-9, "Con tasa 0.01 no debería superar a la red bien entrenada; revisa los parámetros"
print("Precisión con tasa 0.01:", round(precision_lenta, 3))

**Tu respuesta (edita esta celda):** …

## 8. Preguntas de reflexión
1. ¿Qué pasaría si quitáramos la ReLU de la capa oculta? ¿Podría la red separar los anillos?
2. ¿Por qué inicializamos los pesos con números aleatorios y no todos a cero?
3. Una neurona con ReLU cuya entrada es siempre negativa "muere". ¿Cómo lo notarías al entrenar?
4. Keras hará todo esto por ti con `model.fit`. ¿Qué parte del proceso sigues sin querer delegar a ciegas?

## 9. Resumen: qué has aprendido
- Una neurona = suma ponderada + sesgo + activación; una capa = muchas neuronas a la vez (una multiplicación de matrices).
- La activación no lineal es lo que permite a la red resolver problemas que un modelo lineal no puede.
- Entrenar = repetir *adelante → medir error → atrás (gradientes) → actualizar pesos*.
- La **tasa de aprendizaje** manda: pequeña = lento, enorme = inestable.
- La comprobación numérica de gradientes es una buena red de seguridad al programar redes a mano.